### This notebook creates weekend, weekday walking + transit isochrones, as well as walking only isochrones. 

### It also cleans the transit lines file and library point data for web map visualizations. 

### The notebook also sets up a hex grid, that is fundamental for the isochrones and population-weighted calculations that will be conducted in DA_data_analysis.ipynb

In [1]:
import sys
import json
import zipfile
import datetime
import numpy as np
import pandas as pd
import geopandas as gpd
import r5py
import h3
from shapely.geometry import Polygon, LineString, shape

sys.argv.append("--max-memory")
sys.argv.append("16G")
print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Set Up Network and Hex Grid

In [2]:
# Uploading Toronto boundary
toronto_bound = gpd.read_file("data/toronto-boundary-wgs84(1)/citygcs_regional_mun_wgs84.shp")
if toronto_bound.crs != 'EPSG:4326':
    toronto_bound = toronto_bound.to_crs('EPSG:4326')
print(f"Boundary loaded: {toronto_bound.crs}")


Boundary loaded: EPSG:4326


In [3]:
# Generating hex grid clipped to Toronto boundary
resolution = 9  # ~200m between hex centres
min_lng, min_lat, max_lng, max_lat = toronto_bound.total_bounds
 
geojson_polygon = {
    "type": "Polygon",
    "coordinates": [[
        [min_lng, min_lat],
        [max_lng, min_lat],
        [max_lng, max_lat],
        [min_lng, max_lat],
        [min_lng, min_lat]
    ]]
}
 
hex_ids = h3.geo_to_cells(geojson_polygon, resolution)
print(f"Generated {len(hex_ids):,} hex cells")
 
hexagons = []
for hex_id in hex_ids:
    boundary = h3.cell_to_boundary(hex_id)
    polygon_coords = [(lng, lat) for lat, lng in boundary]
    hexagons.append({'hex_id': hex_id, 'geometry': Polygon(polygon_coords)})
 
gdf_hex = gpd.GeoDataFrame(hexagons, crs='EPSG:4326')
 
# Clipping hexes to Toronto boundary
toronto_union = toronto_bound.geometry.unary_union
gdf_hex['geometry'] = gdf_hex.geometry.intersection(toronto_union)
gdf_hex = gdf_hex[~gdf_hex.geometry.is_empty].reset_index(drop=True)
gdf_hex['id'] = gdf_hex.index
 
print(f"Clipped to {len(gdf_hex):,} hex cells within Toronto")

gdf_hex[['id', 'hex_id', 'geometry']].to_file('hex_grid.geojson', driver='GeoJSON')
print("Saved hex_grid.geojson")


Generated 11,518 hex cells


/var/folders/cx/qckjdd7d4b783y6xjpc04zxm0000gn/T/ipykernel_71990/1907072130.py:28: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  toronto_union = toronto_bound.geometry.unary_union


Clipped to 6,017 hex cells within Toronto
Saved hex_grid.geojson


In [4]:
# Building transport network + preparing origins and destinations
osm_path  = "data/Toronto.osm.pbf"
gtfs_path = "data/Complete GTFS.zip"
 
print("Building transport network (this takes a few minutes)...")
transport_network = r5py.TransportNetwork(osm_path, [gtfs_path])
print("Network ready!")
 
# Origins: hex centroids in WGS84
origins_projected = gdf_hex.to_crs('EPSG:32617')  # UTM 17N for accurate centroids
centroids = origins_projected.geometry.centroid.to_crs('EPSG:4326')
 
origins = gpd.GeoDataFrame({'id': gdf_hex.index, 'geometry': centroids}, crs='EPSG:4326')
print(f"Origins: {len(origins):,} hex centroids")
 
# Destinations: library locations
libraries_raw = gpd.read_file("data/tpl-branch-general-information - 4326.geojson")
libraries_raw = libraries_raw[libraries_raw.geometry.notna()].reset_index(drop=True)
 
destinations = gpd.GeoDataFrame({
    'id': libraries_raw.index,
    'geometry': libraries_raw.geometry.representative_point()
}, crs='EPSG:4326')
print(f"Destinations: {len(destinations):,} libraries")
 


Building transport network (this takes a few minutes)...


Network ready!
Origins: 6,017 hex centroids
Destinations: 101 libraries


In [5]:
# Converting r5py results → GeoJSON with time_bucket property for isochrone map visualization
def results_to_geojson(results, hex_polygons, output_path):
    min_times = (
        results
        .groupby('from_id')['travel_time']
        .min()
        .reset_index()
        .rename(columns={'from_id': 'id', 'travel_time': 'minutes'})
    )
 
    merged = hex_polygons.merge(min_times, on='id', how='left')
    merged['minutes'] = merged['minutes'].fillna(60).clip(upper=60)
 
    def bucket(m):
        if m < 15:   return 'under_15'
        elif m < 30: return '15_to_30'
        else:        return 'over_30'
 
    merged['time_bucket'] = merged['minutes'].apply(bucket)
 
    out = merged[['geometry', 'time_bucket', 'minutes']].copy()
    out = out[out.geometry.notna() & ~out.geometry.is_empty]
    if out.crs != 'EPSG:4326':
        out = out.to_crs('EPSG:4326')
 
    out.to_file(output_path, driver='GeoJSON')
    print(f"\nSaved → {output_path}")
    print(f"  under_15 : {(out.time_bucket == 'under_15').sum():,}")
    print(f"  15_to_30 : {(out.time_bucket == '15_to_30').sum():,}")
    print(f"  over_30  : {(out.time_bucket == 'over_30').sum():,}")
 


## 2. Generate Isochrones

Transit uses a 30-minute departure window (10:00–10:30); walking uses a single 10:00 departure since walk times don't depend on schedule.

In [6]:
# Walk + Transit, weekday (Tuesday June 8 2026, 10:00–10:30 window)
print("Running: walk + transit, weekday")
results_weekday = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=destinations,
    transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
    departure=datetime.datetime(2026, 6, 8, 10, 0, 0),
    departure_time_window=datetime.timedelta(minutes=30),
)
print(f"Done — {len(results_weekday):,} calculations")
results_to_geojson(results_weekday, gdf_hex, "isochrones_transit_weekday_window.geojson")

Running: walk + transit, weekday
Done — 607,717 calculations

Saved → isochrones_transit_weekday_window.geojson
  under_15 : 1,397
  15_to_30 : 3,392
  over_30  : 1,228


In [7]:
# Walk + Transit, weekend (Saturday June 13 2026, 10:00–10:30 window)
print("Running: walk + transit, weekend")
results_weekend = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=destinations,
    transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
    departure=datetime.datetime(2026, 6, 13, 10, 0, 0),
    departure_time_window=datetime.timedelta(minutes=30),
)
print(f"Done — {len(results_weekend):,} calculations")
results_to_geojson(results_weekend, gdf_hex, "isochrones_transit_weekend_window.geojson")

Running: walk + transit, weekend
Done — 607,717 calculations

Saved → isochrones_transit_weekend_window.geojson
  under_15 : 1,380
  15_to_30 : 3,409
  over_30  : 1,228


In [8]:
# Walking only (Tuesday June 8 2026, 10:00 — schedule irrelevant)
print("Running: walk only isochrones")
results_walk = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=destinations,
    transport_modes=[r5py.TransportMode.WALK],
    departure=datetime.datetime(2026, 6, 8, 10, 0, 0),
)
print(f"Done — {len(results_walk):,} calculations")
results_to_geojson(results_walk, gdf_hex, "isochrones_walk.geojson")

Running: walk only isochrones


Done — 607,717 calculations

Saved → isochrones_walk.geojson
  under_15 : 1,191
  15_to_30 : 2,254
  over_30  : 2,572


## 3. Export Library Points

In [9]:
libraries_export = libraries_raw[['BranchName', 'Address', 'geometry']].copy()
libraries_export = libraries_export[libraries_export.geometry.notna()].reset_index(drop=True)
libraries_export['geometry'] = libraries_export.geometry.representative_point()
libraries_export.to_file("libraries.geojson", driver='GeoJSON')
print(f"Saved libraries.geojson — {len(libraries_export)} branches")

Saved libraries.geojson — 101 branches


## 4. Patch Snapping Artifacts

Some hexes return implausibly low travel times because the origin centroid snapped to a nearby road or transit stop. This cell identifies hexes whose travel time is far too low for their straight-line distance to the nearest library and resets them to a plausible walk time. Genuine near-library hexes are left alone.

In [10]:
WALK_SPEED_M_PER_MIN = 3600 / 60   # 3.6 km/h = 60 m/min

libs = gpd.read_file("libraries.geojson").to_crs("EPSG:26917")
lib_points = list(libs.geometry)

def bucket(m):
    if m < 15:   return "under_15"
    elif m < 30: return "15_to_30"
    else:        return "over_30"

for fname in ["isochrones_walk.geojson",
              "isochrones_transit_weekday_window.geojson",
              "isochrones_transit_weekend_window.geojson"]:
    gj = json.load(open(fname))
    fixed = 0
    for f in gj["features"]:
        mins = f["properties"]["minutes"]
        if mins is None or mins >= 5:
            continue
        # Straight-line distance from hex centroid to nearest library
        centroid = gpd.GeoSeries(
            [shape(f["geometry"]).centroid], crs="EPSG:4326"
        ).to_crs("EPSG:26917").iloc[0]
        min_dist = min(centroid.distance(lp) for lp in lib_points)
        # Minimum plausible walk time (allow 50% slack for hex extent)
        plausible = (min_dist / WALK_SPEED_M_PER_MIN) * 0.5
        if mins < plausible - 2:
            new_min = round(min_dist / WALK_SPEED_M_PER_MIN)
            f["properties"]["minutes"] = float(new_min)
            f["properties"]["time_bucket"] = bucket(new_min)
            fixed += 1
    json.dump(gj, open(fname, "w"))
    print(f"{fname}: patched {fixed} artifact hexes")

isochrones_walk.geojson: patched 97 artifact hexes
isochrones_transit_weekday_window.geojson: patched 97 artifact hexes
isochrones_transit_weekend_window.geojson: patched 97 artifact hexes


## 5. Export TTC Rapid Transit Lines

Routes 1 (Yonge-University), 2 (Bloor-Danforth), 4 (Sheppard), 5 (Eglinton LRT), 6 (Finch West LRT), dissolved to one feature per line for clean webmap visualizations.

In [11]:
with zipfile.ZipFile("data/Complete GTFS.zip") as z:
    shapes = pd.read_csv(z.open("shapes.txt"))
    trips  = pd.read_csv(z.open("trips.txt"), low_memory=False)
    routes = pd.read_csv(z.open("routes.txt"))

shape_routes = (
    trips[['shape_id', 'route_id']]
    .drop_duplicates()
    .merge(routes[['route_id', 'route_short_name', 'route_long_name', 'route_type']], on='route_id')
    .drop_duplicates('shape_id')
)

lines = []
for shape_id, group in shapes.sort_values('shape_pt_sequence').groupby('shape_id'):
    coords = list(zip(group['shape_pt_lon'], group['shape_pt_lat']))
    if len(coords) >= 2:
        info = shape_routes[shape_routes['shape_id'] == shape_id]
        lines.append({
            'shape_id':         shape_id,
            'route_short_name': info['route_short_name'].iloc[0] if len(info) else None,
            'route_long_name':  info['route_long_name'].iloc[0]  if len(info) else None,
            'route_type':       info['route_type'].iloc[0]       if len(info) else None,
            'geometry':         LineString(coords)
        })

gdf_routes = gpd.GeoDataFrame(lines, crs='EPSG:4326')

keep_routes = [1.0, 2.0, 4.0, 5.0, 6.0]
main_lines = gdf_routes[gdf_routes['route_short_name'].isin(keep_routes)].copy()
main_lines = main_lines.dissolve(by='route_short_name').reset_index()

print(main_lines[['route_short_name', 'route_long_name', 'route_type']])
print(f"{len(main_lines)} lines total")
main_lines.to_file("ttc_main_lines.geojson", driver='GeoJSON')
print("Saved ttc_main_lines.geojson")

   route_short_name            route_long_name  route_type
0                 1  Line 1 (Yonge-University)           1
1                 2  Line 2 (Bloor - Danforth)           1
2                 4          Line 4 (Sheppard)           1
3                 5            Line 5 Eglinton           0
4                 6          Line 6 Finch West           0
5 lines total
Saved ttc_main_lines.geojson
